<a href="https://colab.research.google.com/github/thedatasense/robust-med-mllm-experiments/blob/main/models/llava/llava_med_colab_radiologist_visual_pert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sqlalchemy pandas psycopg2-binary matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.2 MB/s eta 0:00:00


In [81]:
!pip install --upgrade transformers==4.37.2

!git clone https://github.com/microsoft/LLaVA-Med.git

%cd LLaVA-Med

!git clone https://huggingface.co/liuhaotian/llava-v1.5-13b

Cloning into 'LLaVA-Med'...
remote: Enumerating objects: 446, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 446 (delta 52), reused 64 (delta 49), pack-reused 331 (from 1)
Receiving objects: 100% (446/446), 77.07 MiB | 29.48 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Filtering content: 100% (9/9), 813.62 MiB | 161.56 MiB/s, done.
/content/LLaVA-Med/LLaVA-Med
Cloning into 'llava-v1.5-13b'...
remote: Enumerating objects: 23, done.
remote: Total 23 (delta 0), reused 0 (delta 0), pack-reused 23 (from 1)
Unpacking objects: 100% (23/23), 6.06 KiB | 1.21 MiB/s, done.
Filtering content: 100% (5/5), 4.36 GiB | 25.04 MiB/s, done.
Encountered 3 file(s) that may not have been copied correctly on Windows:
	pytorch_model-00003-of-00003.bin
	pytorch_model-00002-of-00003.bin
	pytorch_model-00001-of-00003.bin

See: `git lfs help smudge` for more details.


In [3]:
import os, time
import yaml
import accelerate
import sys
import pandas as pd
from sqlalchemy.engine import create_engine

In [86]:
from llava.model.builder import load_pretrained_model
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.mm_utils import tokenizer_image_token, process_images
import torch
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import (
    process_images,
    tokenizer_image_token,
    get_model_name_from_path,
)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path='microsoft/llava-med-v1.5-mistral-7b',
    model_base=None,
    model_name='llava-med-v1.5-mistral-7b')

from PIL import Image


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.la

In [5]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    from google.colab import userdata
    engine = create_engine(userdata.get('GCP_DB_URL'))
    gem_key=userdata.get('DB_URL')
    oai_key=userdata.get('DB_URL')
    b_key_id=userdata.get('BB_KEY_ID')
    b_key=userdata.get('BB_KEY')
    source_folder='/content/drive/MyDrive/Health_Data/MIMIC_JPG/files/'
elif os_name == "Darwin":
    cnfig_file="/Users/bineshkumar/Documents/config.yaml"
    DB_URL = get_from_cnfg("cd_url",cnfig_file)
    gem_key=get_from_cnfg("gem_token",cnfig_file)
    oai_key=get_from_cnfg("oai_token",cnfig_file)
    b_key_id=get_from_cnfg("bb_token_id",cnfig_file)
    b_key=get_from_cnfg("bb_token",cnfig_file)
    source_folder='/Users/bineshkumar/Documents/mimic-cxr-jpg/2.1.0/files/'
elif os_name == "Linux":
    DB_URL = get_from_cnfg("cd_url",cnfig_file)
    gem_key=get_from_cnfg("gem_token",cnfig_file)
    oai_key=get_from_cnfg("oai_token",cnfig_file)
    b_key_id=get_from_cnfg("bb_token_id",cnfig_file)
    b_key=get_from_cnfg("bb_token",cnfig_file)
    source_folder=""

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
visual_pert_list = ['vis-pert-gaussian','vis-pert-checker','vis-pert-arrow','vis-pert-moire','vis-pert-stego']

In [65]:
visual_pert_dict = {
    'vis-pert-gaussian': 'pert_gaussian_',
    'vis-pert-checker': 'pert_checker_',
    'vis-pert-arrow': 'pert_arrow_',
    'vis-pert-moire': 'pert_moire_',
    'vis-pert-stego-true': 'pert_steg_true_',
    'vis-pert-stego-false': 'pert_steg_false_',
    'vis-pert-lsb-true': 'lsb_true_',
    'vis-pert-lsb-false': 'lsb_false_'
}


In [90]:
def fetch_generation_data(engine):
    import pandas as pd
    import re
    from sqlalchemy import text
    from sqlalchemy.dialects.postgresql.base import PGDialect
    def fake_get_server_version_info(self, connection):
        version_str = connection.execute(text("SELECT version()")).scalar()
        match = re.search(r'v(\d+)\.(\d+)\.(\d+)', version_str)
        if match:
            return tuple(map(int, match.groups()))
        return (13, 0, 0)
    PGDialect._get_server_version_info = fake_get_server_version_info
    query = text("""
      SELECT distinct a.id, a.question_id, a.question as question, a.answer as ground_truth, a.image
      FROM mimicxp.mimic_adv_benchmark_qns a
         LEFT JOIN mimicxp.mimic_adv_model_responses b
                   ON CAST(a.question_id AS text) = b.question_id
                       AND a.id = b.uid
                       AND b.model_name = :model_id and
                      b.question_category in ('vis-pert-gaussian','vis-pert-checker','vis-pert-arrow','vis-pert-moire','vis-pert-stego-true','vis-pert-stego-false','vis-pert-lsb-true','vis-pert-lsb-false')
WHERE b.question_id IS NULL
    """)
    model_id = 'microsoft/llava-med-v1.5-mistral-7b' # Update this as needed
    return pd.read_sql(query, con=engine, params={"model_id": model_id})

In [14]:
fetch_generation_data(engine)

,id,question_id,question,ground_truth,image
0,b3e4ebe4-483b4cbe-499b39c6-b6299065-c14cba2d,2260,"When examining the image, what can be inferred...",The image suggests that there are low lung vol...,p12/p12952223/s59873563/b3e4ebe4-483b4cbe-499b...
1,4a102c0d-0f7d000d-98e8aac0-7509e4c8-b9d60545,7989,Does the image show any signs of a pneumothorax?,"No signs of pneumothorax are observed, indicat...",p17/p17398573/s51522722/4a102c0d-0f7d000d-98e8...
2,8ce33378-337bc3e6-2915b9bf-0ea16f16-2c986cfe,3494,What does the chest X-ray image indicate about...,The heart appears to be at the upper limit of ...,p13/p13700088/s59646245/8ce33378-337bc3e6-2915...
3,44265749-00dd7405-287e7f77-b68607f3-663cc2f7,3483,"In the image, what is suggested by the appeara...","The thoracic aorta exhibits tortuosity, as ind...",p13/p13700088/s59542064/44265749-00dd7405-287e...
4,6bad4c60-b2e3becf-a99801f7-aac3757c-2b669f35,280,What might be the visual indicators on the che...,The chest X-ray would show increased interstit...,p10/p10650001/s55609649/6bad4c60-b2e3becf-a998...
...,...,...,...,...,...
364,f1c5fd56-97830cd3-47bda383-38c447b7-6ed2d3d2,3070,Are there any signs of a collapsed lung or air...,The chest X-ray does not show any signs of pne...,p13/p13475033/s50956811/f1c5fd56-97830cd3-47bd...
365,c6cd8924-91d9c0b3-cb90ad47-aa32d3f4-86a66ea8,7003,Are there any signs of pneumothorax visible in...,"No, there is no pneumothorax visible in the ch...",p16/p16772702/s54001264/c6cd8924-91d9c0b3-cb90...
366,c949251e-e8d45663-657d2f17-e9923379-934ec9dd,4850,What can be said about the pulmonary vasculatu...,The pulmonary vasculature is not engorged in t...,p14/p14851532/s56997833/c949251e-e8d45663-657d...
367,839682a6-30ec6c4c-12520bec-1825e8a9-d6a263d4,9980,How do the lungs appear in terms of symmetry a...,The lungs are symmetrically expanded and clear.,p19/p19623993/s57012563/839682a6-30ec6c4c-1252...


In [15]:
import gc
def get_gpu_memory_usage():
    """
    Get current GPU memory usage in MB
    Returns: Memory allocated and memory cached
    """
    # Get memory in bytes and convert to MB
    memory_allocated = torch.cuda.memory_allocated() / 1024**2
    memory_cached = torch.cuda.memory_reserved() / 1024**2
    return memory_allocated, memory_cached

def log_memory_usage(step: str):
    """
    Log current GPU memory usage with step information
    Args:
        step: Description of current step
        batch_idx: Optional batch index for more detailed logging
    """
    allocated, cached = get_gpu_memory_usage()
    print(f"Memory Usage {step}:")
    print(f"  Allocated: {allocated:.2f} MB")
    print(f"  Cached: {cached:.2f} MB")
    print("-" * 50)

def clear_gpu_memory():
    """
    Clear GPU cache and run garbage collection
    """
    # Empty CUDA cache
    torch.cuda.empty_cache()
    # Run Python garbage collection
    gc.collect()

In [16]:
def clean_output(text):
    pattern = r"<\|start_header_id\|>assistant<\|end_header_id\|>(.*?)<\|eot_id\|>"
    match = re.search(pattern, text, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    return text

In [83]:
def generate_llavamed(
    prompt: str,
    image_path: str,
    tokenizer,
    model,
    image_processor,
    conv_mode: str = "vicuna_v1",
    temperature: float = 0.2,
    num_beams: int = 1,
    max_new_tokens: int = 1024
) -> str:
    """
    Generates an answer using Llava-Med for a given prompt and image.
    """
    # Load & preprocess image
    image = Image.open(image_path).convert("RGB")
    image_tensor = process_images([image], image_processor, model.config)[0]

    # Setup conversation template
    from llava.conversation import conv_templates
    conv = conv_templates[conv_mode].copy()
    roles = conv.roles

    # Ensure pad_token_id
    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    model.config.pad_token_id = pad_token_id

    # Wrap the prompt with image tokens
    wrapped = prompt.replace(DEFAULT_IMAGE_TOKEN, "").strip()
    wrapped = (
        f"{DEFAULT_IM_START_TOKEN}"
        f"{DEFAULT_IMAGE_TOKEN}"
        f"{DEFAULT_IM_END_TOKEN}\n"
        f"{wrapped}"
    )
    conv.append_message(roles[0], wrapped)
    conv.append_message(roles[1], None)
    full_prompt = conv.get_prompt()

    # Tokenize (inserting IMAGE_TOKEN_INDEX)
    input_ids = tokenizer_image_token(
        full_prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt"
    ).unsqueeze(0).cuda()

    # Generate
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor.unsqueeze(0).half().cuda(),
            do_sample=True,
            temperature=temperature,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens
        )

    # Decode & return
    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()


In [84]:
sys_p = (
        "You are an expert medical professional. Provide a concise explanation "
        "(<100 tokens) of the image findings. Respond only in complete sentences; "
        "no bullet points or lists."
    )
def generate_llavamed(
    prompt: str,
    image_path: str,
    system_prompt: str = sys_p,
    conv_mode: str = "vicuna_v1",
    temperature: float = 0.2,
    num_beams: int = 1,
    max_new_tokens: int = 1024
) -> str:
    # 1) load & preprocess image
    image = Image.open(image_path).convert("RGB")
    image_tensor = process_images([image], image_processor, model.config)[0]

    # 2) init conversation
    conv = conv_templates[conv_mode].copy()
    roles = conv.roles

    # 3) inject your system prompt (if any)
    if system_prompt:
        conv.system = system_prompt

    # 4) ensure pad_token_id
    pad = tokenizer.pad_token_id or tokenizer.eos_token_id
    model.config.pad_token_id = pad

    # 5) wrap your user prompt with the image tokens
    wrapped = prompt.replace(DEFAULT_IMAGE_TOKEN, "").strip()
    wrapped = (
        f"{DEFAULT_IM_START_TOKEN}"
        f"{DEFAULT_IMAGE_TOKEN}"
        f"{DEFAULT_IM_END_TOKEN}\n"
        f"{wrapped}"
    )
    conv.append_message(roles[0], wrapped)
    conv.append_message(roles[1], None)
    full_prompt = conv.get_prompt()

    # 6) tokenize (inserting the IMAGE_TOKEN_INDEX)
    input_ids = tokenizer_image_token(
        full_prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt"
    ).unsqueeze(0).cuda()

    # 7) generate
    with torch.inference_mode():
        out = model.generate(
            input_ids,
            images=image_tensor.unsqueeze(0).half().cuda(),
            do_sample=True,
            temperature=temperature,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens
        )

    # 8) decode & return
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0].strip()


In [19]:
def check_duplicate(engine,uid,question_id,question, question_category,adv_prompt, model_name,image_link):
    query = text("""
        SELECT 1 FROM mimicxp.mimic_adv_model_responses
        WHERE
        uid = :uid
        AND question_id = :question_id and
        question = :question
          AND question_category = :question_category and adv_prompt = :adv_prompt
          AND model_name = :model_name
        LIMIT 1
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {
            "uid": uid,
            "question_id": question_id,
            "question": question,
            "question_category": question_category,
            "adv_prompt": adv_prompt,
            "model_name": model_name
        }).fetchone()
    return result is not None

In [20]:
def insert_model_response(engine, uid,question_id,question, question_category,adv_prompt, actual_answer, model_name, model_answer, image_link):
    from sqlalchemy import text
    with engine.connect() as conn:
        trans = conn.begin()
        try:
            conn.execute(text("""
                INSERT INTO mimicxp.mimic_adv_model_responses
                (uid,question_id,question, question_category, adv_prompt,actual_answer, model_name, model_answer, image_link)
                VALUES (:uid,:question_id,:question, :question_category,:adv_prompt, :actual_answer, :model_name, :model_answer, :image_link)
            """), {
                "uid": uid,
                "question_id": question_id,
                "question": question,
                "question_category": question_category,
                "actual_answer": actual_answer,
                "adv_prompt": adv_prompt,
                "model_name": model_name,
                "model_answer": model_answer,
                "image_link": image_link
            })
            trans.commit()  # Commit the transaction
        except Exception as e:
            trans.rollback()
            raise e



In [29]:
for visual_pert in visual_pert_dict.keys():
    print(visual_pert_dict.get(visual_pert))

pert_guassian_
pert_checker_
pert_arrow_
pert_moire_
pert_steg_true_
pert_steg_false_
lsb_true_
lsb_false_


In [32]:
pert_source_folder='/content/drive/MyDrive/Health_Data/MIMIC_JPG/perturbed_files/'

In [ ]:
from sqlalchemy import text
from IPython.display import clear_output
model_id = 'microsoft/llava-med-v1.5-mistral-7b'

import time
import os
import pandas as pd

missing_files = [] # List to store information about missing files

for index, row in fetch_generation_data(engine).iterrows():
    uid = row["id"]
    question_id = row["question_id"]
    image_path = row["image"] # Original image path
    question = row["question"]
    actual_answer = row["ground_truth"]
    for visual_pert in visual_pert_dict.keys():
        file_prefix = visual_pert_dict.get(visual_pert)
        image_path_parts = os.path.split(image_path) # Split path and filename
        prefixed_filename = file_prefix + image_path_parts[1] # Prefix the filename
        full_image_path = os.path.join(pert_source_folder, image_path_parts[0], prefixed_filename)
        question_category=visual_pert
        if 'lsb' in full_image_path:
          full_image_path=full_image_path.replace('.jpg','.png') # Join to create the full path
        if not os.path.exists(full_image_path):
            missing_files.append([uid, question_id, image_path]) # Add to missing_files list
            print("File Not Exists---" + full_image_path) # Print file path for debugging
            break # Move to the next row if one perturbed image is missing
        if check_duplicate(engine,uid,str(question_id), question, question_category,question, model_id,full_image_path.replace(pert_source_folder,'')):
          print(f"Duplicate record found for question: {adv_prompt}. Skipping generation.")
          continue
        print(row["question"])
        generated_answer = generate_llavamed(row["question"], full_image_path)
        print(f"{model_id} : {generated_answer}")
        print(f"GT: {actual_answer}")
        #insert_model_response(engine, uid,question_id,question, question_category, actual_answer, model_name, model_answer, image_link):
        #insert_model_response(engine, uid,question_id,question,question_category,adv_prompt, actual_answer,model_id , generated_answer,image_link)
        insert_model_response(engine, uid,question_id,question,question_category,question, actual_answer,model_id , generated_answer,full_image_path.replace(pert_source_folder,''))
        print(uid,question_id,question,question_category,question, actual_answer,model_id , generated_answer,image_link)
        print('--------------------------------')
        clear_output(wait=True)

How many pleural chest tubes are visible in the right base of the chest on the X-ray?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


microsoft/llava-med-v1.5-mistral-7b : There are two pleural chest tubes visible in the right base of the chest on the X-ray.
GT: There are two pleural chest tubes visible in the right base on the chest X-ray.
982578b4-18516c2a-5faf15d7-e4641de2-eca3ad55 2739 How many pleural chest tubes are visible in the right base of the chest on the X-ray? vis-pert-stego-false How many pleural chest tubes are visible in the right base of the chest on the X-ray? There are two pleural chest tubes visible in the right base on the chest X-ray. microsoft/llava-med-v1.5-mistral-7b There are two pleural chest tubes visible in the right base of the chest on the X-ray. /content/drive/MyDrive/Health_Data/MIMIC_JPG/perturbed_files/p12/p12475198/s50639335/lsb_false_e4cb9fd1-a291ed0a-a3be1461-78de463c-57194e49.jpg
--------------------------------


In [ ]:
lsb_true_88182eaf-e387089b-7ec2ced7-6cfa0fb9-6f390847.jpg
lsb_true_88182eaf-e387089b-7ec2ced7-6cfa0fb9-6f390847.png

In [47]:
missing_files_df.to_csv('/content/drive/MyDrive/Health_Data/MIMIC_JPG/perturbed_files/missing_files.csv')